In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
NOTEBOOK = "/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/notebooks/02_jobs/20_ingest_run"
JOB_NAME = "oil-shock-ingest"

existing = [j for j in w.jobs.list(name=JOB_NAME)]
if existing:
    job_id = existing[0].job_id
    print(f"Job already exists: {job_id}")
else:
    job = w.jobs.create(
        name=JOB_NAME,
        tasks=[jobs.Task(
            task_key="ingest",
            notebook_task=jobs.NotebookTask(notebook_path=NOTEBOOK, source=jobs.Source.WORKSPACE),
            timeout_seconds=3600,
        )],
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 7,21 * * ?",
            timezone_id="Asia/Dubai",
            pause_status=jobs.PauseStatus.PAUSED,
        ),
        email_notifications=jobs.JobEmailNotifications(on_failure=["zulhaq2@gmail.com"]),
        max_concurrent_runs=1,
    )
    job_id = job.job_id
    print(f"Created job {JOB_NAME}: {job_id} (schedule paused)")

run = w.jobs.run_now(job_id=job_id)
print(f"Test run started: run_id {run.run_id}")
print("Watch it under Jobs & Pipelines > oil-shock-ingest > Runs")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
job_id = 970079484069909

w.jobs.update(job_id=job_id, new_settings=jobs.JobSettings(
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 7,21 * * ?",
        timezone_id="Asia/Dubai",
        pause_status=jobs.PauseStatus.UNPAUSED,
    )
))

s = w.jobs.get(job_id).settings.schedule
print(f"Schedule: {s.quartz_cron_expression} ({s.timezone_id}) — {s.pause_status.value}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
job_id = 970079484069909
NB = "/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/notebooks/02_jobs"

w.jobs.reset(job_id=job_id, new_settings=jobs.JobSettings(
    name="oil-shock-ingest",
    tasks=[
        jobs.Task(
            task_key="ingest",
            notebook_task=jobs.NotebookTask(notebook_path=f"{NB}/20_ingest_run", source=jobs.Source.WORKSPACE),
            timeout_seconds=3600,
        ),
        jobs.Task(
            task_key="dbt_build",
            depends_on=[jobs.TaskDependency(task_key="ingest")],
            run_if=jobs.RunIf.ALL_DONE,
            notebook_task=jobs.NotebookTask(notebook_path=f"{NB}/22_dbt_build", source=jobs.Source.WORKSPACE),
            timeout_seconds=3600,
        ),
    ],
    schedule=jobs.CronSchedule(quartz_cron_expression="0 0 7,21 * * ?", timezone_id="Asia/Dubai",
                               pause_status=jobs.PauseStatus.UNPAUSED),
    email_notifications=jobs.JobEmailNotifications(on_failure=["zulhaq2@gmail.com"]),
    max_concurrent_runs=1,
))

s = w.jobs.get(job_id).settings
print("Tasks:", [t.task_key for t in s.tasks], "| Schedule:", s.schedule.quartz_cron_expression, s.schedule.pause_status.value)

run = w.jobs.run_now(job_id=job_id)
print("Test run started:", run.run_id)